# ClipNova: test the open-source video models on a free GPU

1. **Turn on a GPU.** Colab: *Runtime -> Change runtime type -> T4 GPU*. Kaggle: *Settings -> Accelerator -> GPU T4* and **Internet: On**.
2. Run the cells in order. The first run downloads each model (several GB) and can take 5-15 minutes.
3. At the end you get one short video per model that fits this GPU, plus render times and memory use.

A free T4 (16 GB) runs **LTX-Video** and **CogVideoX-2B**. **Wan 2.2** needs a 24 GB bf16 GPU and is skipped automatically.

In [ ]:
# 1) Is there a GPU?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Get the code. The repo is private, so paste a GitHub token (read access to CLIPNOVA.AI) when asked. It is not stored.
import getpass, subprocess, os
token = getpass.getpass("GitHub token: ")
subprocess.run(["rm", "-rf", "CLIPNOVA.AI"])
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", "feat/m1-foundation",
                    f"https://{token}@github.com/Sanjanag11/CLIPNOVA.AI.git"], capture_output=True, text=True)
print("cloned" if r.returncode == 0 else r.stderr.replace(token, "***"))
os.chdir("CLIPNOVA.AI/workers/inference")

In [ ]:
# 3) Install (torch is already on Colab/Kaggle)
!pip install -q -e ".[gpu]"
!ffmpeg -version | head -1

In [ ]:
# 4) Make a short clip with every model this GPU can run. Edit the prompt, seconds or steps as you like.
!python -m clipnova_worker.smoke --seconds 2 --steps 20 --prompt "A fluffy orange cat rides a skateboard down a sunny street, smooth camera, cinematic lighting"

In [ ]:
# 5) Watch the results
import glob
from IPython.display import Video, display
for f in sorted(glob.glob("smoke_out/*.mp4")):
    print(f)
    display(Video(f, embed=True, width=360))

In [ ]:
# 6) Download them: Colab -> Files panel (left) -> smoke_out. Or zip:
!zip -qr clips.zip smoke_out && ls -la clips.zip

## What to tell me
- Which model looks best to you, and whether any looks unusable.
- The **render time** and **peak GPU memory** lines printed in step 4.
- Any error text, copied as-is. A failed model prints `FAILED` with the reason.